# 1/2 — Prepare ViHSD

Run this private Kaggle notebook once. Enable Internet, accept access to `uitnlp/vihsd` on Hugging Face, and add the Kaggle secret `HF_TOKEN`. GPU is not required. After completion, use **Save Version**, then attach this notebook's output as an Input to notebook 2. Do not publish the prepared data.

In [ ]:
# CPU-only data preparation; this must run before any optional CUDA import.
import os, sys, subprocess
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface_hub', 'pandas', 'pyarrow',
], check=True)
OUTPUT_DIR = Path('/kaggle/working/vihsd_prepared')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUTPUT_DIR)

In [ ]:
import hashlib, json
import pandas as pd
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as exc:
    raise RuntimeError(
        'Create a Kaggle Secret named HF_TOKEN and enable it for this notebook.'
    ) from exc
if not HF_TOKEN or not HF_TOKEN.startswith('hf_'):
    raise RuntimeError('HF_TOKEN is missing or has an invalid format.')
MODEL_ID, DATASET_ID = 'uitnlp/visobert', 'uitnlp/vihsd'
api = HfApi(token=HF_TOKEN)
try:
    account = api.whoami()['name']
    model_revision = api.model_info(MODEL_ID).sha
    dataset_revision = api.dataset_info(DATASET_ID).sha
except Exception as exc:
    raise RuntimeError(
        'Hugging Face authentication/access failed. Create a new Read token, '
        'accept the ViHSD gated-dataset terms in the browser, and update HF_TOKEN.'
    ) from exc
print({'huggingface_account': account, 'access': 'verified'})
source_files = {split: hf_hub_download(
    repo_id=DATASET_ID, repo_type='dataset', filename=filename,
    revision=dataset_revision, token=HF_TOKEN,
) for split, filename in {
    'train': 'train.csv', 'validation': 'dev.csv', 'test': 'test.csv'
}.items()}
expected_counts = {'train': 24048, 'validation': 2672, 'test': 6680}
output_files = {}
empty_text_rows = {}
for split, source_path in source_files.items():
    frame = pd.read_csv(source_path)
    assert len(frame) == expected_counts[split], (split, len(frame))
    assert {'free_text', 'label_id'} <= set(frame.columns), frame.columns.tolist()
    assert set(frame['label_id'].astype(int).unique()) <= {0, 1, 2}
    text_values = frame['free_text'].fillna('').astype(str)
    empty_text_rows[split] = int(text_values.str.strip().eq('').sum())
    prepared = pd.DataFrame({
        'sample_id': [f'{split}_{index:08d}' for index in range(len(frame))],
        'text': text_values,
        'labels': frame['label_id'].astype(int),
        'split': split,
    })
    target = OUTPUT_DIR / f'{split}.parquet'
    prepared.to_parquet(target, index=False)
    output_files[split] = target
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
manifest = {
    'artifact': 'vihsd_prepared_v1',
    'model_id': MODEL_ID, 'model_revision': model_revision,
    'dataset_id': DATASET_ID, 'dataset_revision': dataset_revision,
    'schema': {'text': 'text', 'label': 'labels',
               'label2id': {'CLEAN': 0, 'OFFENSIVE': 1, 'HATE': 2}},
    'data_quality': {
        'empty_text_policy': 'preserve row and replace missing text with empty string',
        'empty_text_rows': empty_text_rows,
    },
    'splits': {split: {
        'file': path.name, 'rows': expected_counts[split], 'sha256': sha256(path),
    } for split, path in output_files.items()},
}
(OUTPUT_DIR / 'vihsd_manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps({
    'model_revision': model_revision, 'dataset_revision': dataset_revision,
    'rows': expected_counts, 'empty_text_rows': empty_text_rows,
    'output': str(OUTPUT_DIR),
}, indent=2))

## Next step

Choose **Save Version → Save & Run All**. In notebook `01_train_evaluate_visobert.ipynb`, choose **Add Input** and attach this notebook's saved output. Keep both notebooks private because the output contains gated ViHSD text.